# Week 12 — Connect to Atlas and recover a ticket

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/week_12/materials/notebooks/05_mongodb_atlas_recovery.ipynb)

[View on GitHub](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_12/materials/notebooks/05_mongodb_atlas_recovery.ipynb)
**Standalone Week 12 recovery lesson:** the Thanksgiving single-meeting schedule belongs to Week 13. We connect to a real Atlas Free cluster, inspect five synthetic records, save a file, restore into a different collection, and repair one wrong value. This notebook does not switch to `mongomock` when a connection fails.

**Start:** save your own copy in Drive. Use the CPU runtime. Open Atlas in another browser tab. The instructor demonstrates Cells 01–06; you complete the one-line change in Cell 07, run Cell 08, and write the explanation in Cell 09. The individual repair takes about **20 minutes once Atlas is connected**; allow class time for setup first.

**Submit one viewable Colab link in Brightspace.** Keep the output, your changed line, and your explanation in your Drive copy. No `.ipynb` attachment, separate written response, exported file, or Atlas screenshot is required.

## What is running where?

- **MongoDB** stores documents. One document below represents one repair request.
- **Atlas** hosts MongoDB. Your collection remains there when the Colab runtime ends.
- **Colab** runs Python on a remote computer. **PyMongo** sends its database requests to Atlas.
- A **database** groups collections; a **collection** groups documents. We use one teaching database with a different source and restore collection for each run.

Run numbered cells in order. Setup cells can be expanded with **Show code**; student changes, write filters, and checks remain visible. No live election, paid backup, or validator exercise is required. Validation remains optional reference, with no promise that it must be completed next week.


## Before Cell 01 — prepare Atlas

1. Sign into your own Atlas account and open an existing **Free** cluster, or create a Free cluster if needed. Do not upgrade, add a card, or enable a backup subscription.
2. Under **Database & Network Access**, create/use a **database user** with `readWrite` on database **`cst4714_week12`**. Use a database user to connect your program to Atlas. Do not share the instructor's credentials with students. On a shared project, the instructor must provide appropriately isolated database access; do not grant broader roles just to bypass an error.
3. Choose your cluster's **Connect → Drivers → Python** connection string. Replace its username/password placeholders with your database-user credentials. Special characters in a URI username/password must be percent-encoded. Keep the complete URI out of notebook cells, messages, screenshots, and saved outputs.
4. Run Cell 02 and add **that runtime IPv4 address `/32`** as a temporary Atlas IP entry. Your laptop browser's **Add Current IP** can authorize the wrong computer. Use the shortest appropriate expiry. Do not allow access from everywhere.
5. In Colab, the optional **Secrets** panel can store `MONGODB_URI`; allow this notebook to read it. If it is unavailable, Cell 03 asks for the URI in a hidden input field. People viewing your shared notebook need their own credentials to rerun it.

Record the error category and check cluster availability, credentials, and database permissions. If blocked, record the error category with credentials removed. Label any output from the instructor's demonstration.


## 01 — Install the driver
Run this supplied setup cell. PyMongo connects your Python program to the MongoDB server.


In [ ]:
#@title Cell 01 — Install PyMongo
%pip -q install "pymongo>=4.13,<5"


## 02 — Find the computer Atlas must allow
Run the cell, then add the displayed `/32` address in Atlas with a temporary expiry. It contacts the public IP lookup service **ipify** without database credentials. A new Colab runtime can have a different address.


In [ ]:
#@title Cell 02 — Show the Colab runtime IPv4 address
from ipaddress import IPv4Address
from urllib.request import urlopen
try:
    with urlopen("https://api.ipify.org", timeout=10) as response:
        runtime_ipv4 = str(IPv4Address(response.read().decode().strip()))
    print("Temporary Atlas IP access-list entry:", runtime_ipv4 + "/32")
except Exception:
    raise RuntimeError("Could not read the runtime IPv4 address. Retry the lookup; do not open access from everywhere.") from None


## 03 — Connect, then ask the server to respond
Run this cell only after the network entry is active. A client object alone does not prove a connection: **ping** asks the server for a response. This cell does not print your URI. Read/write permission is checked by the next cell.


In [ ]:
#@title Cell 03 — Connect to actual Atlas and ping
from getpass import getpass
from urllib.parse import urlsplit
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from pymongo.errors import OperationFailure, ServerSelectionTimeoutError

if "client" in globals() and client is not None:
    client.close()
client = None
uri = None
try:
    try:
        from google.colab import userdata
        uri = userdata.get("MONGODB_URI")
    except Exception:
        uri = getpass("Atlas driver URI (hidden): ")
    host = (urlsplit(uri).hostname or "").lower()
    if not uri.startswith("mongodb+srv://") or not host.endswith(".mongodb.net"):
        raise ValueError("Use the mongodb+srv Atlas Drivers URI, with placeholders replaced.")
    if "<" in uri or ">" in uri:
        raise ValueError("Replace the username/password placeholders before connecting.")
    client = MongoClient(uri, tls=True, tlsInsecure=False, tz_aware=True,
                         server_api=ServerApi("1", strict=True, deprecation_errors=True),
                         serverSelectionTimeoutMS=10000, timeoutMS=10000,
                         appname="CST4714-Week12-Classroom")
    client.admin.command("ping")
    print("Atlas responded to ping. This is a real PyMongo server connection.")
except ValueError:
    if client is not None: client.close()
    client = None
    raise RuntimeError("The Atlas URI is incomplete or is not the standard Drivers URI. Check it privately; never paste it into a code cell.") from None
except OperationFailure:
    if client is not None: client.close()
    client = None
    raise RuntimeError("Atlas refused authentication or authorization. Check the database user, password, and teaching-database role.") from None
except Exception:
    if client is not None: client.close()
    client = None
    raise RuntimeError("Atlas connection failed. Check the runtime /32 rule, deployment state, URI, DNS, and TLS. No mock database has been substituted.") from None
finally:
    uri = None


## Data reference — read this before the first write

Every row below is one complete **application record**. MongoDB adds `_id`, its unique identifier, on insertion. That generated value varies by run and is retained in the saved file. `ticket_id` is our stable course identifier, protected by a unique index in the source collection. There are no joins or separate requester records in this fixture.

| ticket_id (integer) | status (text) | priority (text) | subject (text) | opened_at (UTC BSON date) |
|---|---|---|---|---|
| 1001 | open | high | Streetlight dark near bus stop | 2026-02-01 23:10 |
| 1002 | in_progress | medium | Missed recycling pickup | 2026-02-02 15:45 |
| 1003 | resolved | urgent | Low water pressure | 2026-02-03 12:05 |
| 1004 | new | low | Broken bench slat | 2026-02-04 17:20 |
| 1005 | resolved | high | Overflowing corner bin | 2026-02-05 14:00 |

`status` names the workflow state; this fixture uses new/open/in_progress/resolved, and the broader course also uses closed. `priority` is low/medium/high/urgent. All five fields are present and non-null here. A missing field and an explicit null are different states; neither occurs in these five records. Python displays a BSON date as a timezone-aware `datetime`. A date-looking string is still text. Read the saved types during the restore.


## 04 — Write five records and read them back
Run the setup, then open Atlas **Browse Collections / Data Explorer** and choose `cst4714_week12` plus the printed source collection. Find ticket 1004. Its subject should be **Broken bench slat** in both places. This write/read proves more than ping.

Rerunning this cell creates new collections. It never drops an existing database or resets another run. The code requests majority acknowledgment; a successful normal write does not test failure or prove a recovery guarantee.


In [ ]:
#@title Cell 04a — Supplied definitions and isolated collection names
from datetime import datetime, timezone
from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import uuid4
from copy import deepcopy
import json
from bson import json_util
from bson.json_util import CANONICAL_JSON_OPTIONS
from pymongo import ReadPreference
from pymongo.read_concern import ReadConcern
from pymongo.write_concern import WriteConcern

assert client is not None, "Connect successfully in Cell 03 first."
DATABASE_NAME = "cst4714_week12"
run_id = uuid4().hex[:12]
SOURCE_COLLECTION = "source_" + run_id
RESTORE_COLLECTION = "restore_" + run_id
assert SOURCE_COLLECTION != RESTORE_COLLECTION
practice_database = client[DATABASE_NAME]
source_tickets = practice_database.get_collection(
    SOURCE_COLLECTION, write_concern=WriteConcern(w="majority"),
    read_preference=ReadPreference.PRIMARY, read_concern=ReadConcern("majority"))
restore_tickets = practice_database.get_collection(
    RESTORE_COLLECTION, write_concern=WriteConcern(w="majority"),
    read_preference=ReadPreference.PRIMARY, read_concern=ReadConcern("majority"))
assert source_tickets.count_documents({}) == 0, "Choose a fresh run; do not overwrite existing records."
source_documents = [
 {"ticket_id":1001,"status":"open","priority":"high","subject":"Streetlight dark near bus stop","opened_at":datetime(2026,2,1,23,10,tzinfo=timezone.utc)},
 {"ticket_id":1002,"status":"in_progress","priority":"medium","subject":"Missed recycling pickup","opened_at":datetime(2026,2,2,15,45,tzinfo=timezone.utc)},
 {"ticket_id":1003,"status":"resolved","priority":"urgent","subject":"Low water pressure","opened_at":datetime(2026,2,3,12,5,tzinfo=timezone.utc)},
 {"ticket_id":1004,"status":"new","priority":"low","subject":"Broken bench slat","opened_at":datetime(2026,2,4,17,20,tzinfo=timezone.utc)},
 {"ticket_id":1005,"status":"resolved","priority":"high","subject":"Overflowing corner bin","opened_at":datetime(2026,2,5,14,0,tzinfo=timezone.utc)},
]


In [ ]:
#@title Cell 04 — Write five documents and read them back
try:
    source_tickets.create_index("ticket_id", unique=True, name="unique_ticket_id")
    inserted = source_tickets.insert_many(deepcopy(source_documents))
    source_rows = list(source_tickets.find({}).sort("ticket_id",1))
    assert inserted.acknowledged and len(source_rows) == 5
except Exception:
    raise RuntimeError("The fixture write/read did not complete. Check readWrite access to cst4714_week12; do not grant wider access just to bypass this error.") from None
print("Database:", DATABASE_NAME)
print("Source collection:", SOURCE_COLLECTION)
print("Restore collection:", RESTORE_COLLECTION)
print("Acknowledged:", inserted.acknowledged, "| Source count:", len(source_rows))
for row in source_rows: print(row["ticket_id"], row["subject"])


## 05 — Save the documents with their date types
Run this cell. Ordinary JSON parsing sees the `$date` representation as a dictionary; the BSON-aware reader reconstructs a Python datetime. The export contains the small classroom dataset, which stays fixed during the exercise.

The file initially lives in the temporary Colab runtime. Use **Retain a copy** below to download it before ending the runtime; saving a notebook does not save its runtime files. The download contains only these synthetic tickets. A real recovery copy needs storage and retention outside the system being recovered.


In [ ]:
#@title Cell 05 — Save and inspect Extended JSON
JSON_OPTIONS = CANONICAL_JSON_OPTIONS.with_options(tz_aware=True)
practice_files = TemporaryDirectory(prefix="cst4714_atlas_recovery_")
EXPORT_FILE = Path(practice_files.name) / "tickets.extended.json"
saved_rows = list(source_tickets.find({}).sort("ticket_id",1))
EXPORT_FILE.write_text(json_util.dumps(saved_rows, json_options=JSON_OPTIONS, indent=2), encoding="utf-8")
export_text = EXPORT_FILE.read_text(encoding="utf-8")
saved_documents = json_util.loads(export_text, json_options=JSON_OPTIONS)
saved_by_id = {row["ticket_id"]: row for row in saved_documents}
print("Saved records:", len(saved_documents))
print("Ordinary JSON reader:", type(json.loads(export_text)[0]["opened_at"]).__name__)
print("BSON-aware reader:", type(saved_documents[0]["opened_at"]).__name__)
print("Saved ticket 1004:", saved_by_id[1004]["subject"])


### Retain a copy outside the runtime
Download the file for the recovery demonstration. In local Jupyter, copy the printed artifact path to a location you retain.


In [ ]:
#@title Cell 05b — Download the synthetic recovery file
try:
    from google.colab import files
    files.download(str(EXPORT_FILE))
except ImportError:
    print("Copy this file outside the temporary runtime:", EXPORT_FILE)


## 06 — Restore elsewhere, then expose a missed error
The source is untouched. This cell restores into the separate target and changes **only target ticket 1004**. It prints a failed values check intentionally; this is the problem you will repair. The source and saved artifact still say **Broken bench slat**.

Read the update filter before running: it identifies one ticket in the separate restore collection. No database or collection is deleted. Counts, IDs, and types can pass while the subject is wrong.


### Trace the records before this experiment

![Record relationships for Week 12](https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/main/week_12/illustrations/lesson_relationships.png)

The [complete data reference](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_12/DATA_REFERENCE.md) provides the same relationships as readable text.


In [ ]:
#@title Cell 06 — Restore a separate target and inspect the wrong value
assert SOURCE_COLLECTION != RESTORE_COLLECTION
if restore_tickets.count_documents({}) == 0:
    restore_tickets.insert_many(deepcopy(saved_documents))
else:
    assert set(r["ticket_id"] for r in restore_tickets.find({})) == set(saved_by_id), "Unexpected target records; start a fresh run."
changed = restore_tickets.update_one(
    {"ticket_id": 1004}, {"$set": {"subject": "Wrong subject in restored copy"}})
assert changed.matched_count == 1

def check_restored():
    actual = list(restore_tickets.find({}).sort("ticket_id",1))
    expected = sorted(saved_documents, key=lambda r:r["ticket_id"])
    checks = {
      "Count is five": len(actual) == 5,
      "Ticket IDs match": [r["ticket_id"] for r in actual] == [r["ticket_id"] for r in expected],
      "Opening dates are datetimes": all(isinstance(r["opened_at"],datetime) for r in actual),
      "Complete document values match": actual == expected,
    }
    for label, passed in checks.items(): print(label + ":", "PASS" if passed else "FAIL")
    return all(checks.values())
check_restored()
print("Saved subject:", saved_by_id[1004]["subject"])
print("Target subject:", restore_tickets.find_one({"ticket_id":1004})["subject"])


## 07 — Your change: recover ticket 1004 from the artifact
Replace **`None`** with a lookup in `saved_by_id` for ticket **1004**. Do not type the correct subject into the update: recover the saved record. Run the cell. All four checks should now pass. The filter and replacement stay visible so you can explain exactly what is written.


In [ ]:
#@title Cell 07 — Select the saved document and repair one ticket
replacement = None  # Your change: look up ticket 1004 in saved_by_id.
if replacement is None:
    print("INCOMPLETE: select the saved document for ticket 1004, then rerun Cell 07.")
else:
    assert replacement["ticket_id"] == 1004, "Repair the assigned ticket, not another record."
    repair = restore_tickets.replace_one({"ticket_id":1004}, deepcopy(replacement), upsert=False)
    print("Matched:", repair.matched_count, "Modified:", repair.modified_count)
    assert repair.matched_count == 1
    assert check_restored(), "The target still differs from the saved artifact."


## 08 — Repeat the repair and inspect Atlas again
Expected: **matched 1, modified 0, five tickets**, and all values match. Repeating a replacement of the same saved document does not append a sixth record. Refresh the restore collection in Atlas and check ticket 1004. Repeating the repair leaves the same final document.


In [ ]:
#@title Cell 08 — Repeat and verify the same repair
if replacement is None:
    print("INCOMPLETE: finish Cell 07 before the repeat check.")
else:
    repeat = restore_tickets.replace_one({"ticket_id":1004}, deepcopy(replacement), upsert=False)
    print("Matched:",repeat.matched_count,"Modified:",repeat.modified_count,"Count:",restore_tickets.count_documents({}))
    assert (repeat.matched_count,repeat.modified_count,restore_tickets.count_documents({})) == (1,0,5)
    assert check_restored()
    assert source_tickets.find_one({"ticket_id":1004})["subject"] == "Broken bench slat"
    print("Source unchanged; repair is repeatable. No paid backup or failover test was used.")


## 09 — Your explanation and submission
Replace the prompts below with **3–4 sentences** in this text cell:

1. Explain what showed a real Atlas connection and successful write/read.
2. Explain why the first checks missed the wrong subject and how the artifact repaired it.
3. Explain why replicated copies alone would not recover an earlier value, and name one thing the document file does not restore (for example an index, validator, or application configuration).

**My explanation:** …

Save your copy and outputs. Choose **Share** and give your instructor viewer access. If permitted, choose Anyone with the link → Viewer; otherwise explicitly share with the instructor. Submit your **own saved Colab URL** in Brightspace and check that the instructor can open it. Do not submit the starter URL or an `.ipynb` attachment. No separate Lab 1 response is due for this single-session Week 12.


## After you save — close the connection
This does not delete your collections or downloaded artifact. Keep the temporary Atlas network rule's expiry; remove access when no longer needed. Any collection cleanup must be limited to the exact names printed in your run and done after retaining your evidence. There is no automatic database deletion cell.


In [ ]:
#@title Cell 10 — Close the client without deleting data
if client is not None:
    client.close()
    print("Client closed. Atlas collections and your downloaded artifact were not deleted.")


## Industry/Career Connection — verify the recovery path
GitLab's published account of its January 2017 PostgreSQL outage describes an accidental deletion and problems with available recovery mechanisms. The incident illustrates the importance of testing a recovery path. A database administrator should retain a usable artifact, rehearse restoration, compare meaningful values, and explain any lost interval. Our five-ticket exercise practices those checks; it does not reproduce GitLab's production incident or establish production recovery targets.

Source: [GitLab's incident postmortem](https://about.gitlab.com/blog/postmortem-of-database-outage-of-january-31/). No additional assignment.

## Sources and limits
- [Connect with an Atlas driver](https://www.mongodb.com/docs/atlas/driver-connection/): database users, TLS, and network access.
- [PyMongo connection targets](https://www.mongodb.com/docs/languages/python/pymongo-driver/current/connect/connection-targets/): creating a client and confirming with ping.
- [Atlas IP access lists](https://www.mongodb.com/docs/atlas/security/ip-access-list/): single addresses and temporary entries.
- [Free cluster limits](https://www.mongodb.com/docs/atlas/reference/free-shared-limitations/): managed three-node deployment; no primary-failover test.
- [Atlas backup options](https://www.mongodb.com/docs/atlas/backup-restore-cluster/): current Free-cluster backup subscription is billed and private preview. We do not enable it.

This is a small logical document recovery exercise. The exercise exports a fixed set of documents while the source is idle. Index and validator reconstruction is optional reference and is not required this week.
